In [15]:
import os
import io
from pathlib import Path
import re
import time
from urllib.parse import urljoin, urlparse
import pandas as pd
import requests
from bs4 import BeautifulSoup


In [16]:
# Define your target directory structure for the local RAG pipeline

BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'Ingestion_AnnualReport' else Path.cwd().resolve().parents[2] 
print(Path.cwd().resolve().parents[2] )
print(f"[✓] Base directory resolved at: {BASE_DIR.resolve()}")
BASE_DIR = BASE_DIR / 'Data' / 'AnnualReport'
BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"[✓] Directory initialized at: {BASE_DIR.resolve()}")

C:\
[✓] Base directory resolved at: C:\Repo\StockAnalysis_AI
[✓] Directory initialized at: C:\Repo\StockAnalysis_AI\data\AnnualReport


In [17]:
class ScreenerScraper:
    def __init__(self, base_dir):
        self.base_dir = Path(base_dir)
        self.headers = {
            'User-Agent': (
                'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                'AppleWebKit/537.36 (KHTML, like Gecko) '
                'Chrome/120.0.0.0 Safari/537.36'
            )
        }

    def download_annual_reports(self, symbol: str, years_limit: int = 4):
        url = f"https://www.screener.in/company/{symbol}/"
        print(f"\n[+] Processing: {symbol} -> {url}")

        try:
            response = requests.get(url, headers=self.headers, timeout=15)
            if response.status_code != 200:
                print(f"[-] HTTP Error {response.status_code} for {symbol}")
                return
        except Exception as e:
            print(f"[-] Request failed for {symbol}: {e}")
            return

        soup = BeautifulSoup(response.text, 'html.parser')

        doc_section = soup.find('section', id='documents')
        if not doc_section:
            print(f"  [-] No <section id='documents'> found for {symbol}")
            return

        container_div = doc_section.find(
            'div',
            class_=lambda c: c and 'documents' in c and 'annual-reports' in c
        )
        if not container_div:
            print(f"  [-] Annual reports container div not found for {symbol}")
            return

        ul_element = container_div.find('ul')
        if not ul_element:
            print(f"  [-] No <ul> found inside annual reports div for {symbol}")
            return

        downloaded_count = 0

        for a_tag in ul_element.find_all('a', href=True):
            if downloaded_count >= years_limit:
                break

            link_text = a_tag.get_text(strip=True)
            match = re.search(r'Annual\s+Report\s+(20\d{2})', link_text, re.IGNORECASE)

            if match:
                year = match.group(1)
                file_url = urljoin("https://www.screener.in", a_tag['href'])

                target_dir = self.base_dir / symbol / year
                target_dir.mkdir(parents=True, exist_ok=True)

                # FIX: Strip out query parameters (e.g. ?Pname=...) from the filename path
                parsed_url = urlparse(file_url)
                clean_path = parsed_url.path
                raw_filename = os.path.basename(clean_path)

                if not raw_filename.lower().endswith('.pdf'):
                    file_name = f"{symbol}_Annual_Report_{year}.pdf"
                else:
                    file_name = raw_filename

                file_path = target_dir / file_name

                if not file_path.exists():
                    print(f"  [>] Downloading {symbol} Annual Report {year}...")
                    try:
                        response = requests.get(file_url, headers=self.headers, timeout=30)
                        if response.status_code == 200:
                            with open(file_path, 'wb') as f:
                                f.write(response.content)
                            print(f"  [✓] Saved: {file_path}")
                            downloaded_count += 1
                        else:
                            print(f"  [-] Download failed (HTTP {response.status_code}) for year {year}")
                    except Exception as e:
                        print(f"  [-] Error downloading report for {year}: {e}")
                else:
                    print(f"  [i] Annual Report {year} already exists at {file_path}")
                    downloaded_count += 1
# if __name__ == "__main__":
#     TEST_SYMBOL = "INDOMIM"
#     BASE_DIR = "./Data/AnnualReport"

#     scraper = ScreenerScraper(base_dir=BASE_DIR)
#     scraper.download_annual_reports(symbol=TEST_SYMBOL, years_limit=4)

In [18]:
# Set your base download directory
# BASE_DIR = Path("./Data/AnnualReport")

# Initialize scraper and run test for a single ticker
if __name__ == "__main__":
    run_for_single_ticker = False  # Set to True to run for a single ticker, False to process from CSV
    if run_for_single_ticker:
        ticker = "MTARTECH"  # Change to any NSE symbol (e.g., RELIANCE, INFOSYS, HDFCBANK)
        
        scraper = ScreenerScraper(base_dir=BASE_DIR)
        
        print(f"Starting test download for: {ticker}")
        scraper.download_annual_reports(symbol=ticker, years_limit=4)
        print("\nTest completed.")
    else:
        df_fo = pd.read_csv("symbols.csv")
        print(df_fo)
        scraper = ScreenerScraper(base_dir=BASE_DIR)
        for index, row in df_fo.iterrows():
            # 2-second polite delay between requests
            time.sleep(5)
            symbol = row['symbol']
            print(f"\n[+] Fetching data for {symbol}..." )
            scraper.download_annual_reports(symbol=symbol, years_limit=4)
            print(f"\n[✓] Completed processing for {symbol}.")

       symbol
0        SIKA
1  ASTRAMICRO
2       PARAS
3   IDEAFORGE
4      ZENTEC
5  ZENSARTECH
6        EIEL

[+] Fetching data for SIKA...

[+] Processing: SIKA -> https://www.screener.in/company/SIKA/
  [>] Downloading SIKA Annual Report 2026...
  [✓] Saved: C:\Repo\StockAnalysis_AI\Data\AnnualReport\SIKA\2026\fafb317a-6f44-4854-8458-5ccbb4f4e294.pdf
  [>] Downloading SIKA Annual Report 2025...
  [✓] Saved: C:\Repo\StockAnalysis_AI\Data\AnnualReport\SIKA\2025\4dbc8ee1-a85d-4a5c-823b-5fd937159598.pdf
  [>] Downloading SIKA Annual Report 2024...
  [✓] Saved: C:\Repo\StockAnalysis_AI\Data\AnnualReport\SIKA\2024\6ab51399-f3fa-4b46-b824-4e37dac43c66.pdf
  [>] Downloading SIKA Annual Report 2023...
  [✓] Saved: C:\Repo\StockAnalysis_AI\Data\AnnualReport\SIKA\2023\60465ab0-6c42-4b3a-b5f2-791a55f51436.pdf

[✓] Completed processing for SIKA.

[+] Fetching data for ASTRAMICRO...

[+] Processing: ASTRAMICRO -> https://www.screener.in/company/ASTRAMICRO/
  [>] Downloading ASTRAMICRO Annual Re